# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"

In [4]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [5]:
import chromadb
from chromadb.utils import embedding_functions

chroma_client = chromadb.PersistentClient(path="chromadb")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction()
collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [6]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

@tool
def retrieve_game(query: str) -> list:

    results = collection.query(
        query_texts=[query],
        n_results=5
    )

    return results["metadatas"][0]

#### Evaluate Retrieval Tool

In [7]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> dict:
    import json
    llm = LLM()
    messages = [
        SystemMessage(content="""Your task is to evaluate if the documents are enough to respond the query.
        You MUST respind ONLY with a valid JSON object, nothing else.
        No markdown, no code blocks, no extra text.
        Foramt: {"useful": true, "description": "your explanation here"}"""),
        UserMessage(content=f"Question: {question}\n\nDocuments: {retrieved_docs}")
    ]

    response = llm.invoke(messages)

    content = response.content.strip()
    content = content.replace("```json", "").replace("```","").strip()

    if not content:
        return {"useful": False, "description": "Could not evaluate retrieval."}

    return json.loads(content)

#### Game Web Search Tool

In [8]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

@tool
def game_web_search(question: str) -> str:
    from tavily import TavilyClient
    client = TavilyClient(TAVILY_API_KEY)
    response = client.search(question)
    return response["results"] 

### Agent

In [9]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

agent = Agent(
    model_name="gpt-4o-mini",
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    instructions="""You are UdaPlay, an AI research agent for the video game industry.
    You help users find information about video games.
    
    Follow this process:
    1. Use retrive_game to search the vector database
    2. Use evaluate_retrieval to assess if results are good enough
    3. If not useful, use game_web_search to find more information
    4. Return a structured, helpul response"""
)

In [10]:
# TODO: Invoke your agent

questions=[
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X realeased for Playstation 5?"
]

for question in questions:
    try:
        run = agent.invoke(question)
        final_state = run.get_final_state()
        messages = final_state["messages"]

        for msg in reversed(messages):
            if isinstance(msg, AIMessage):
                print(f"Q: {question}")
                print(f"A: {msg.content}")
                break
    except Exception as e:
        print(f"Q: {question}")
        print(f"Error: {e}\n")


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Q: When Pokémon Gold and Silver was released?
A: Pokémon Gold and Silver were released in 1999 for the Game Boy Color. These games are part of the second generation of Pokémon, introducing new regions, Pokémon, and gameplay mechanics.
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Q: Which one was the first 3D platformer Mario game?
A: The first 3D platformer Mario game is **Super Mario 64**, which was released in 1996 for the Nintendo 64. This groundbreaking game set ne

### (Optional) Advanced

In [11]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes